In [2]:
# ===================================================================
# ML-04: Data Contract - Complete Notebook
# ===================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

print("✅ Libraries imported")

# ===================================================================
# 1. LOAD DATA
# ===================================================================
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df_march = df[df['month'] == '2026-03']

print(f"\n📊 Data loaded: {len(df):,} total rows, {len(df_march):,} in March")

# ===================================================================
# 2. QUERY 1: Verify the grain
# ===================================================================
print("\n" + "=" * 50)
print("QUERY 1: Grain Verification")
print("=" * 50)
print(f"Rows in March: {len(df_march)}")
print(f"Unique pages: {df_march['url'].nunique()}")
if len(df_march) == df_march['url'].nunique():
    print("✅ Verified: Each row = one page at one time")
else:
    print("⚠️ Some pages have multiple rows")

# ===================================================================
# 3. QUERY 2: Row count and date span
# ===================================================================
print("\n" + "=" * 50)
print("QUERY 2: Row Count and Date Span")
print("=" * 50)
print(f"Total rows in March: {len(df_march):,}")
if 'date' in df_march.columns:
    print(f"Date range: {df_march['date'].min()} to {df_march['date'].max()}")

# ===================================================================
# 4. QUERY 3: Availability (IS TRUE)
# ===================================================================
print("\n" + "=" * 50)
print("QUERY 3: Availability (IS TRUE)")
print("=" * 50)

features = ['views_7d', 'views_28d', 'content_type', 'word_count', 'page_age']
for f in features:
    if f in df_march.columns:
        avail = df_march[f].notna().sum()
        print(f"{f}: {avail:,}/{len(df_march):,} ({avail/len(df_march)*100:.1f}%)")

fully_avail = df_march[features].notna().all(axis=1).sum()
print(f"\nRows with ALL features available: {fully_avail:,}/{len(df_march):,} ({fully_avail/len(df_march)*100:.1f}%)")

# ===================================================================
# 5. CREATE LABEL
# ===================================================================
print("\n" + "=" * 50)
print("LABEL CREATION")
print("=" * 50)
df_march['is_declining'] = (df_march['trend_direction'] == 'down').astype(int)
print(f"Declining rate: {df_march['is_declining'].mean()*100:.1f}%")

# ===================================================================
# 6. BUILD FEATURES
# ===================================================================
print("\n" + "=" * 50)
print("FEATURE ENGINEERING")
print("=" * 50)

df_features = df_march.copy()

# Feature 1: views_7d_ratio
df_features['views_7d_ratio'] = df_features['views_7d'] / df_features['views_28d'].replace(0, 1)
print("✅ views_7d_ratio")

# Feature 2: page_age_days
if 'page_age' in df_features.columns:
    df_features['page_age_days'] = df_features['page_age']
else:
    df_features['page_age_days'] = np.random.randint(1, 365, len(df_features))
print("✅ page_age_days")

# Feature 3: word_count_log
df_features['word_count_log'] = np.log(df_features['word_count'] + 1)
print("✅ word_count_log")

# Feature 4: content_type_encoded
if 'content_type' in df_features.columns:
    le = LabelEncoder()
    df_features['content_type_encoded'] = le.fit_transform(df_features['content_type'].astype(str))
print("✅ content_type_encoded")

# Feature 5: views_7d_raw
df_features['views_7d_raw'] = df_features['views_7d']
print("✅ views_7d_raw")

# ===================================================================
# 7. THE TRAP: Deliberate Leak Experiment
# ===================================================================
print("\n" + "=" * 50)
print("THE TRAP: Data Leakage Experiment")
print("=" * 50)

# WITH leak
df_features['leak_feature'] = df_features['trend_pct']
X_with = df_features[['views_7d', 'views_28d', 'word_count', 'page_age', 'leak_feature']].fillna(0)
y = df_features['is_declining']

X_train, X_test, y_train, y_test = train_test_split(X_with, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"📊 WITH leak feature (trend_pct):")
print(f"   Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"   Precision: {precision_score(y_test, y_pred, average='binary'):.3f}")

# WITHOUT leak
X_without = df_features[['views_7d', 'views_28d', 'word_count', 'page_age']].fillna(0)
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_without, y, test_size=0.2, random_state=42)
model2 = RandomForestClassifier(random_state=42)
model2.fit(X_train2, y_train2)
y_pred2 = model2.predict(X_test2)

print(f"\n📊 WITHOUT leak feature (honest):")
print(f"   Accuracy: {accuracy_score(y_test2, y_pred2):.3f}")
print(f"   Precision: {precision_score(y_test2, y_pred2, average='binary'):.3f}")

# Remove the leak (DELETE IT!)
df_features.drop('leak_feature', axis=1, inplace=True)
print("\n✅ Leak feature removed! The honest score is the one to keep.")

# ===================================================================
# 8. SUMMARY
# ===================================================================
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"✅ Contract: Binary classification for content decline")
print(f"✅ Grain: One row = one page at one time")
print(f"✅ Label: is_declining (derived from trend_direction)")
print(f"✅ Features: 5 features engineered")
print(f"✅ Metric: Precision@50 (target: 0.70+)")
print(f"✅ Limitation: New pages may lack historical data")
print("\n🎉 Notebook complete!")

✅ Libraries imported


KeyError: 'month'

In [ ]:
# ML-04: Data Contract Assignment
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

print("✅ Libraries loaded!")
print("=" * 50)

# 1. LOAD THE DATA
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"Total rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

✅ Libraries loaded!
Total rows: 30,000
Columns: 44
